In [189]:
import pandas as pd
import numpy as np
import json
import csv
from docx import Document
from bs4 import BeautifulSoup
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [190]:
def parse_csv(file_path: str) -> str:
    """
    Same reasoning as JSON — each row gets flattened into one line so the
    chunker's paragraph splitter treats a row as a natural unit rather than
    an arbitrary character slice through the middle of a row.
    """
    lines = []
    with open(file_path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            parts = [f"{key}: {value}" for key, value in row.items()]
            lines.append(", ".join(parts))
    return "\n".join(lines)

In [191]:
def parse_docx(file_path: str) -> str:
    """
    Requires: pip install python-docx
    Paragraphs are joined with double newlines so the chunker's paragraph
    splitter treats each Word paragraph as its own unit, same as it would
    for a PDF page or a markdown paragraph.
    """
    doc = Document(file_path)
    paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    return "\n\n".join(paragraphs)

In [192]:
def parse_html(file_path: str) -> str:
    """
    Requires: pip install beautifulsoup4
    Strips tags/scripts/styles down to visible text. get_text(separator="\n\n")
    approximates paragraph breaks from block-level tags so the chunker still
    has structure to split on, rather than one giant run-on string.
    """
    with open(file_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    for tag in soup(["script", "style"]):
        tag.decompose()

    text = soup.get_text(separator="\n\n")
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n\n".join(lines)

In [193]:
def parse_json(file_path: str) -> str:
    """
    JSON is structured data, not prose — chunking it by raw character count
    would slice through braces and produce garbage for embedding. Instead we
    flatten each record into a sentence-like line. Adjust the flattening
    template below to fit your actual schema.
    """
    with open(file_path, "r", encoding="utf-8") as f:
        records = json.load(f)

    if isinstance(records, dict):
        records = [records]

    lines = []
    for record in records:
        parts = [f"{key}: {value}" for key, value in record.items()]
        lines.append(", ".join(parts))
    return "\n".join(lines)

In [194]:
def parse_markdown(file_path: str) -> str:
    """
    Headers are kept as-is (not stripped) — the chunker's paragraph splitter
    treats blank lines as boundaries, so headers stay glued to the content
    that follows them.
    """
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read().strip()

In [195]:
def parse_pdf(file_path: str) -> str:
    
    reader = PdfReader(file_path)
    pages = []
    for page in reader.pages:
        text = page.extract_text() or ""
        text = text.strip()
        if text:
            pages.append(text)
    return "\n\n".join(pages)

In [196]:
def parse_txt(file_path: str) -> str:
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read().strip()

In [197]:
PARSERS = {
    "csv": parse_csv,
    "docx": parse_docx,
    "html": parse_html,
    "json": parse_json,
    "md": parse_markdown,
    "pdf": parse_pdf,
    "txt": parse_txt,
}

In [198]:
def parse_document(file_path: str, file_format: str) -> str:
    parser = PARSERS.get(file_format)
    if parser is None:
        raise ValueError(f"No parser registered for format: {file_format}")
    return parser(file_path)

In [199]:
sample_data_csv  = parse_document("Sample Files/sample.csv", "csv")
sample_data_docx = parse_document("Sample Files/sample.docx", "docx")
sample_data_html = parse_document("Sample Files/sample.html", "html")
sample_data_json = parse_document("Sample Files/sample.json", "json")
sample_data_md   = parse_document("Sample Files/sample.md", "md")
sample_data_pdf  = parse_document("Sample Files/sample.pdf", "pdf")
sample_data_txt  = parse_document("Sample Files/sample.txt", "txt")


In [200]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    add_start_index=True,
    separators=["\n\n", "\n", ". ", " ", ""],
)

In [201]:
chunks_csv  = splitter.split_text(sample_data_csv)
chunks_docx = splitter.split_text(sample_data_docx)
chunks_html = splitter.split_text(sample_data_html)
chunks_json = splitter.split_text(sample_data_json)
chunks_md   = splitter.split_text(sample_data_md)
chunks_pdf  = splitter.split_text(sample_data_pdf)
chunks_txt  = splitter.split_text(sample_data_txt)

In [202]:
print(chunks_pdf[0])

RockAI Internship Program Internship Learning Plan — AI Engineering 
Internal — RockAI Development Team / TPM Page 1 of 12 
 
YOUR LEARNING PLAN  
AI ENGINEERING TRACK 
 
Personalized Skill Development Plan — RockAI Dev 
 
Intern Name: Sohayl Islam El-Mahdy 
Specialty Track: AI Engineering 
Plan Edition: v2.0 (Detailed Execution) 
Audit Scope: Direct-address Personalized Learning Schedule
